# Iter-SPIN — matched-budget CONTROL (one-shot SPIN at Iter-SPIN's budgets)

Answers: *did iterating help, or did we just zero more weights?* Runs **one-shot** SPIN (single
identification on the UNMODIFIED model, authors' exact rule) at the **same weight budgets** as the
Iter-SPIN snapshots, using two matching strategies in one place:

- **Global-budget** — search a single `p=q` so the total coupled count matches each Iter-SPIN
  snapshot (124/156/219/380 for 0.5B). Lands *near* the target (count is discrete in p). Controls for
  *how many* weights are zeroed; Iter-SPIN may still win via placement OR selection.
- **Per-category (per-matrix)** — give the control Iter-SPIN's exact per-matrix counts, then pick the
  top-k coupled candidates in each matrix by coupling strength. Hits the target **exactly** and also
  matches *placement*, so Iter-SPIN can only win via within-matrix *selection*.

Targets + per-matrix distribution are read automatically from the model's own snapshots
`coupled_iter_{MODEL_TAG}_n*.json` — nothing hand-typed. Importance is scored ONCE (top-K cache),
then both controls re-threshold cheaply. Outputs `coupled_control_{global,percat}_{MODEL_TAG}_n{n}.json`,
which drop into the unchanged Step 2/3/4 exactly like the snapshots.

## Config

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece','huggingface_hub'], check=False)
import random, gc, time, json, glob, re
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ---- Hugging Face Authentication Path ----
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF")
login(token=hf_token, add_to_git_credential=False)

# ---- model lineup: uncomment ONE (must match the model whose Iter-SPIN snapshots you're controlling) ----
MODEL_ID='google/gemma-2-2b-it'                  

USE_FAST=False        # set True for Llama-3/3.2 (fast-only tokenizer)
NSAMPLES=128; TARGET='mlp'; MAXLEN=1024; LAYER_GROUP=4
P0=5e-7               # SPIN's default ratio (start of the global search)
K_CAP=20000          # per-matrix top-K cache size (>= largest top_k the search needs; scale-safe)
RANK='min'           # per-category coupling-strength rank: 'min' or 'prod' of (imp_f, imp_p)
SEED=0; DTYPE=torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_TAG=MODEL_ID.split('/')[-1]
def memline(tag=''):
    s=''.join(f' GPU{d} {torch.cuda.mem_get_info(d)[0]/1e9:.1f}/{torch.cuda.mem_get_info(d)[1]/1e9:.1f}' for d in range(torch.cuda.device_count()))
    print(f'  [{tag}]{s} GiB free')
print('model', MODEL_ID, '| tag', MODEL_TAG, '| K_CAP', K_CAP, '| rank', RANK)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 55.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.2.0 which is incompatible.


model google/gemma-2-2b-it | tag gemma-2-2b-it | K_CAP 20000 | rank min


## Data + model (author sampling, identical to Step 1)

In [2]:
if not os.path.exists('DEAN'):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI45Lab/DEAN.git'], check=True)
DATA='DEAN/src/data'
CSV={'g':f'{DATA}/alpaca_cleaned_no_safety_train.csv',
     'f':f'{DATA}/beaver_train330k_fairness_safe_1k.csv',
     'p':f'{DATA}/beaver_train330k_privacy_safe_1k.csv'}
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]
def make_batches(csv_path, n):
    td=load_dataset('csv', data_files={'train':csv_path}, split='train').shuffle(seed=SEED).select(range(n))
    out=[]
    for i in range(n):
        ep=tokenizer(str(td['prompt'][i]),   return_tensors='pt').input_ids
        er=tokenizer(str(td['response'][i]), return_tensors='pt').input_ids[:,1:]
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={k:make_batches(CSV[k],NSAMPLES) for k in CSV}
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
print('matrices', len(names), '| groups', len(GROUPS), '| batches g/f/p', [len(BATCHES[k]) for k in 'gfp'])

Cloning into 'DEAN'...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model loaded | entry cuda:0
  [loaded] GPU0 12.3/15.6 GPU1 13.5/15.6 GiB free


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

matrices 78 | groups 7 | batches g/f/p [128, 128, 128]


## Score importance ONCE -> per-matrix top-K cache

In [3]:
@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}
    del acc; gc.collect(); torch.cuda.empty_cache(); return imp

CK='/kaggle/working/control_cache_%s.pt'%MODEL_TAG
if os.path.exists(CK):
    CACHE=torch.load(CK); print('loaded cache for', len(CACHE), 'matrices')
else:
    CACHE={}
    model.gradient_checkpointing_enable(); model.config.use_cache=False
    for gi,grp in enumerate(GROUPS):
        gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
        t0=time.time()
        imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
        for n in gnames:
            numel=imp_f[n].numel(); K=min(K_CAP,numel)
            fv,fi=torch.topk(imp_f[n].flatten(),K); pv,pi=torch.topk(imp_p[n].flatten(),K); _,gi_=torch.topk(imp_g[n].flatten(),K)
            CACHE[n]={'rows':imp_f[n].shape[0],'cols':imp_f[n].shape[1],
                      'gi':gi_.int(),'fi':fi.int(),'pi':pi.int(),'fv':fv.float(),'pv':pv.float()}
        del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
        print(f'  group {gi+1}/{len(GROUPS)} cached ({time.time()-t0:.0f}s)')
    model.gradient_checkpointing_disable()
    torch.save(CACHE, CK)
mlp_params=int(sum(CACHE[n]['rows']*CACHE[n]['cols'] for n in CACHE))
print('cache ready | matrices', len(CACHE), '| MLP params', mlp_params)

  group 1/7 cached (108s)
  group 2/7 cached (110s)
  group 3/7 cached (98s)
  group 4/7 cached (80s)
  group 5/7 cached (68s)
  group 6/7 cached (63s)
  group 7/7 cached (54s)
cache ready | matrices 78 | MLP params 1656225792


## Authors' exact selection, re-thresholded from the cache

In [4]:
def coupled_matrix(n, kf, kp, kg):
    # authors' get_set_difference_mask logic (positional mask), reconstructed from cached top-K
    c=CACHE[n]
    ug=torch.unique(c['gi'][:kg]); uf=torch.unique(c['fi'][:kf]); up=torch.unique(c['pi'][:kp])
    m1=~torch.isin(uf,ug); m2=~torch.isin(up,ug)
    L=min(len(m1),len(m2)); inter=m1[:L]&m2[:L]            # authors' positional AND
    return uf[:L][inter]

def coupled_count_at_ratio(r):
    tot=0; per={}
    for n,c in CACHE.items():
        k=int(r*c['rows']*c['cols'])
        if k<1: continue
        if k>len(c['fi']): raise RuntimeError(f'K_CAP too small: need k={k} for {n}; raise K_CAP')
        fi=coupled_matrix(n,k,k,k)
        if len(fi): per[n]=fi; tot+=len(fi)
    return tot, per
print('sanity @ P0=%.1e -> coupled count'%P0, coupled_count_at_ratio(P0)[0])

sanity @ P0=5.0e-07 -> coupled count 235


## Read Iter-SPIN targets (counts + per-matrix distribution) for this model

In [5]:
def _find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
snaps={}
for f in sorted(glob.glob('**/coupled_iter_%s_n*.json'%MODEL_TAG, recursive=True) +
                glob.glob('/kaggle/input/**/coupled_iter_%s_n*.json'%MODEL_TAG, recursive=True)):
    n=int(re.search(r'_n(\d+)\.json$', f).group(1)); snaps[n]=json.load(open(f))
assert snaps, 'no coupled_iter_%s_n*.json found -- add the Iter-SPIN snapshots as input'%MODEL_TAG
TARGETS={n:s for n,s in sorted(snaps.items()) if n>=2}   # n=1 == one-shot SPIN already; skip
print('targets (n -> total coupled):', {n:s['coupled_count'] for n,s in TARGETS.items()})

targets (n -> total coupled): {2: 423, 3: 588, 5: 922, 10: 1596}


## Control A — global budget (single p=q search, count matches ~target)

In [6]:
# grow ratio until we cover the largest target, then bisect per target
maxT=max(s['coupled_count'] for s in TARGETS.values())
r_hi=P0
while coupled_count_at_ratio(r_hi)[0] < maxT: r_hi*=1.6
print('search bracket: [%.2e, %.2e]'%(P0, r_hi))

def search_ratio(target, lo=P0, hi=r_hi, iters=40):
    best=None
    for _ in range(iters):
        mid=(lo+hi)/2; c,per=coupled_count_at_ratio(mid)
        if best is None or abs(c-target)<abs(best[0]-target): best=(c,mid,per)
        if c<target: lo=mid
        else: hi=mid
    return best   # (actual_count, ratio, per_matrix_indices)

global_meta={}
for n,s in TARGETS.items():
    T=s['coupled_count']; c,r,per=search_ratio(T)
    coupled={m:[int(x) for x in v.tolist()] for m,v in per.items()}
    meta={'model':MODEL_ID,'variant':'control-global','matched_to_iter_n':n,'target_count':T,
          'coupled_count':int(sum(len(v) for v in coupled.values())),'ratio_p_q':r,'coupled':coupled}
    json.dump(meta, open(f'/kaggle/working/coupled_control_global_{MODEL_TAG}_n{n}.json','w'))
    global_meta[n]=(T,meta['coupled_count'],r)
    print(f'  n={n:2d}: target {T:4d} -> got {meta["coupled_count"]:4d}  (p=q={r:.3e})  -> coupled_control_global_{MODEL_TAG}_n{n}.json')

search bracket: [5.00e-07, 3.28e-06]
  n= 2: target  423 -> got  432  (p=q=9.773e-07)  -> coupled_control_global_gemma-2-2b-it_n2.json
  n= 3: target  588 -> got  583  (p=q=1.281e-06)  -> coupled_control_global_gemma-2-2b-it_n3.json
  n= 5: target  922 -> got  927  (p=q=1.975e-06)  -> coupled_control_global_gemma-2-2b-it_n5.json
  n=10: target 1596 -> got 1607  (p=q=3.190e-06)  -> coupled_control_global_gemma-2-2b-it_n10.json


## Control B — per-category (exact per-matrix match, ranked by coupling strength)

In [7]:
# per-category: match Iter-SPIN's EXACT per-matrix counts. For each matrix, rank ALL of its weights
# by coupling strength min/prod(imp_f, imp_p) and take exactly k_n. Prefer coupled-&-not-general
# weights (faithful to SPIN); only relax the \\N_g rule to backfill if a matrix lacks enough. Exact by construction.
def candidates_ranked(n):
    c=CACHE[n]
    fmap={int(i):float(v) for i,v in zip(c['fi'].tolist(), c['fv'].tolist())}
    pmap={int(i):float(v) for i,v in zip(c['pi'].tolist(), c['pv'].tolist())}
    common=set(fmap)&set(pmap)                       # weights in top-K of BOTH fairness and privacy
    Sg=set(int(x) for x in c['gi'].tolist())
    strength=(lambda i: fmap[i]*pmap[i]) if RANK=='prod' else (lambda i: min(fmap[i],pmap[i]))
    primary =sorted(common-Sg, key=strength, reverse=True)   # coupled & not general (faithful)
    backfill=sorted(common&Sg, key=strength, reverse=True)   # relaxed: general-important, used only if short
    return primary, backfill

percat_meta={}
for n,s in TARGETS.items():
    dist={m:len(v) for m,v in s['coupled'].items()}          # Iter-SPIN's per-matrix counts
    coupled={}; relaxed=0; missing=0
    for m,kn in dist.items():
        if m not in CACHE: missing+=kn; continue
        primary,backfill=candidates_ranked(m)
        pick=primary[:kn]
        if len(pick)<kn:                                     # backfill to stay EXACT
            need=kn-len(pick); add=backfill[:need]; relaxed+=len(add); pick=pick+add
        coupled[m]=[int(x) for x in pick]
        if len(coupled[m])<kn: missing+=kn-len(coupled[m])   # only if matrix truly can't supply (extremely rare)
    tot=int(sum(len(v) for v in coupled.values()))
    meta={'model':MODEL_ID,'variant':'control-percat','matched_to_iter_n':n,'target_count':s['coupled_count'],
          'coupled_count':tot,'rank':RANK,'relaxed_backfill':int(relaxed),'unfillable':int(missing),'coupled':coupled}
    json.dump(meta, open(f'/kaggle/working/coupled_control_percat_{MODEL_TAG}_n{n}.json','w'))
    percat_meta[n]=(s['coupled_count'],tot,relaxed)
    tag_ok='EXACT' if tot==s['coupled_count'] else f'!={s["coupled_count"]}'
    print(f'  n={n:2d}: target {s["coupled_count"]:4d} -> got {tot:4d} [{tag_ok}] (relaxed \\Ng: {relaxed})  -> coupled_control_percat_{MODEL_TAG}_n{n}.json')

  n= 2: target  423 -> got  423 [EXACT] (relaxed \Ng: 0)  -> coupled_control_percat_gemma-2-2b-it_n2.json
  n= 3: target  588 -> got  588 [EXACT] (relaxed \Ng: 0)  -> coupled_control_percat_gemma-2-2b-it_n3.json
  n= 5: target  922 -> got  922 [EXACT] (relaxed \Ng: 0)  -> coupled_control_percat_gemma-2-2b-it_n5.json
  n=10: target 1596 -> got 1596 [EXACT] (relaxed \Ng: 0)  -> coupled_control_percat_gemma-2-2b-it_n10.json


## Summary

In [8]:
print('='*74); print('  MATCHED-BUDGET CONTROLS for', MODEL_TAG); print('='*74)
print(f'{"iter n":>7}{"iter total":>12}{"global got":>12}{"percat(exact)":>12}{"global p=q":>14}')
for n in TARGETS:
    it=TARGETS[n]['coupled_count']; g=global_meta[n][1]; pc=percat_meta[n][1]; r=global_meta[n][2]
    print(f'{n:>7}{it:>12}{g:>12}{pc:>12}{r:>14.3e}')
print('\nNext: feed each coupled_control_*.json to Step 2/3/4 (as you do the snapshots) and compare')
print('Iter-SPIN(n) vs control-global(n) vs control-percat(n) at MATCHED budget.')
print('  global = matches count (iteration may win via placement or selection)')
print('  percat = matches count + placement (iteration can only win via within-matrix selection)')

  MATCHED-BUDGET CONTROLS for gemma-2-2b-it
 iter n  iter total  global gotpercat(exact)    global p=q
      2         423         432         423     9.773e-07
      3         588         583         588     1.281e-06
      5         922         927         922     1.975e-06
     10        1596        1607        1596     3.190e-06

Next: feed each coupled_control_*.json to Step 2/3/4 (as you do the snapshots) and compare
Iter-SPIN(n) vs control-global(n) vs control-percat(n) at MATCHED budget.
  global = matches count (iteration may win via placement or selection)
  percat = matches count + placement (iteration can only win via within-matrix selection)
